# ═══════════════════════════════════════════════════════════════════════════
# REQUIREMENTS
#
# Example (Tic Tac Toe):
#   1. Two players alternate placing X and O on a 3x3 grid.
#   2. A player wins by completing a row, column, or diagonal.
#   Out of Scope: UI, AI opponent, networking
# ═══════════════════════════════════════════════════════════════════════════

REQUIREMENTS

1. Game starts with an empty board of 7 columns and 6 rows
2. Each player take turns to place a disk of different color (yellow and red)
3. Player chooses which column to drop the disk and the disk will land on the lowest empty spot
4. If player tries to drop disk to a column that is full, action is rejected and player needs to choose another column
5. A player wins by connecting 4 disks of the same color either horizontally, vertically or diagonally.
6. When the board is full and no player get 4 disks align, the game ends in a draw.
7. Players can reset the game anytime. The game will be restarted with an empty board.


FEEDBACK

What went well

You nailed all three core operations — dropping a disc into a chosen column, the disc falling to the lowest available spot, and players alternating turns. These are the backbone of the game and you covered them clearly. 
Your win and draw conditions are spot on, including all three alignment directions (horizontal, vertical, diagonal) and the full-board draw scenario — both of which you confirmed in your clarifying questions. 
You didn't skip error handling, which many candidates do. Calling out that a full-column drop should be rejected and the player must choose again shows solid defensive thinking. 

Minor suggestions

You might consider noting a few things that are explicitly out of scope — like no UI, no move history, and no undo — since you confirmed those in your clarifying questions. Stating scope boundaries briefly helps keep your design focused and signals to the interviewer that you're intentional about what you're building. 

MY CORRECTION

These parts are missing

Error conditions:
1. When player tries to place disc on an invalid column (full, or out of range).
2. Wrong turn - player plays out of turn
3. Invalid state - player tries to place disc when game is not in-progress 

Error handling: 
Return False or throw exception

Out of Scope:
- UI
- keep track of moving history
- Undo action
- support more than 2 players
- concurrent games
- changing board size

# ═══════════════════════════════════════════════════════════════════════════
# ENTITIES & RELATIONSHIPS
#
# Example (Tic Tac Toe):
#   Game, Board, Player
# ═══════════════════════════════════════════════════════════════════════════
<!-- The game contains a board and 2 players. The game orchestrates the flow of the game, keeping track of the players' turn and state of the game (win, draw or in-progress). It will also contain logic to determine if there is a win.
The board keeps track of the state of each spot (empty or occupied), state of the board itself (empty or full) and validate if a disk drop to a column is allowed. It will also expose a function to reset the board (make it empty).
The player will expose a function to return the player's color.  -->

ENTITIES:
Game, board, player

RELATIONSHIPS:
Game -> Board
Game -> Player (2x)

FEEDBACK

What went well

You nailed the three core objects — Game, Board, and Player are exactly the right building blocks for this system. 
It's great that you already started thinking about relationships, showing you're thinking ahead to how these pieces connect. 

Minor suggestions

You might consider adding a DiscColor enum (something like RED and YELLOW) to represent the two different disc types — it's a small but useful thing to have when you're tracking which player owns which disc on the board. 

FEEDBACK

What went well

You clearly identified Game as the orchestrator — explaining that it manages turn order, game state, and win detection shows strong understanding of how the flow is driven. 
Your ownership relationships are solid: Game contains a Board and two Players, and you described what each entity is responsible for, which makes the dependencies easy to follow. 

# ═══════════════════════════════════════════════════════════════════════════
# CLASS DESIGN
#
# Example (Tic Tac Toe):
#   class Game:
#     - board: Board
#     - currentPlayer: Player
#     + makeMove(row, col) -> bool
# ═══════════════════════════════════════════════════════════════════════════
<!-- The game class is the orchestrator here. It contains the board and the 2 players. It also owns switching between the 2 players and implements the rules to determine if the game is won or ends in a draw, as well as reset the game.

The board class owns the grid in which the location of each colored disc is stored. It has function for dropping a disc into a column. If the drop is successful, it will update the grid and return true. However, if the column is already full, it will return false. It has function isOccupiedByColor() so that the caller can check if a spot is taken by a disc of specific color. The game class will use this function while checking for winning conditions. Another function isBoardFull() is used to check if the board has no more available spots. This way, the game class can end the game with a draw. Function emptyBoard() is called to reset the board to empty (initial) state.

The player class is a simple class with a function to return the disc color. -->

class Game:
- board: Board
- player1: Player
- player2: Player
- currentPlayer: Player
- winner: Player?
- state: gameState //enum(won, draw, in_progress)
+ makeMove(col) -> bool
+ getWinner() -> Player
+ getCurrentPlayer() -> Player
+ getGameState() -> gameState
+ reset()

class Board:
- const cols: 7
- const rows: 6
- grid: discColor?[rows][cols]
+ dropDisc(col, discColor) -> int
+ isOccupiedByColor(row, col, discColor) -> bool
+ emptyBoard()
+ isBoardFull() -> bool

class Player:
- color: discColor //enum(red, yellow)
+ Player(discColor)
+ getDiscColor() -> discColor


FEEDBACK

What went well

Your class separation is really clean — Game orchestrates the flow, Board owns the grid, and Player is a simple data holder. That's exactly the right way to think about responsibilities. 
You've got solid state coverage across all three classes: currentPlayer, winner, state on Game, the grid on Board, and color on Player. Each class knows exactly what it needs to know. 
Your verbal explanation of how dropDisc() returns a bool and how isOccupiedByColor() feeds into win checking shows you've thought through how these classes actually collaborate at runtime — that's a great sign. 

What needs improvement

Win detection is the one piece that's missing from your Board class. You have isOccupiedByColor() which could support win checking, but Board doesn't expose a method that actually checks for four in a row — that logic needs to live somewhere, and Board is the right owner since it has direct access to the grid data. 

Minor suggestions

You might consider whether isOccupiedByColor(row, col, color) is the right primitive to expose publicly, or whether it's more of an internal helper used by a win-checking method inside Board — exposing it publicly means Game ends up doing a lot of the win-checking loop logic itself, which leaks grid knowledge upward. 

MY CORRECTION

This parts are missing. 
- Summarize each class about its responsibility (remember SRP) and its boundaries.
- Called out which methods will mutate properties (esp. state) and which ones are read only.

Game Class:



Implementation
Let's add win detection to your Board class. Implement a checkWin(row, col, color) method that checks whether placing a disc at the given position creates four in a row horizontally, vertically, or diagonally. Walk me through the algorithm — how would you structure the four directional checks?

In [4]:
# I have 4 helper functions has4Vertically(), has4Horizontally(), _has4DiagonallyAsc() and _has4DiagonallyDsc() 
# to help checking for consecutive disc colors in different directions.

# has4Vertically() checks the disc color vertically in both directions. has4Horizontally() checks the disc color 
# horizontally in both directions. While _has4DiagonallyAsc() and _has4DiagonallyDsc() check diagonally in 
# ascending and descending directions respectively.

# The checkWin() will call these 4 helper functions. If any one of them returns true, that means there is a win
#  and checkWin() will return true.

# For the edge cases, checkWin() will perform a check on the validity of the input parameters. The helper 
# functions will check for boundary conditions and make sure to exit once the connect-four condition has been met (by using a counter)."    

from enum import Enum
from abc import ABC, abstractmethod

class DiscColor(Enum):
    RED = 1
    YELLOW = 2

class GameState(Enum):
    IN_PROGRESS = 0
    WON = 1
    DRAW = 2
    STOPPED = 3

class Player(ABC):
    def __init__(self, color: DiscColor):
        self._color = color

    @property
    def DiscColor(self):
        return self._color
    
    @abstractmethod
    def getDropCol(self) -> int:
        pass
    
class HumanPlayer(Player):
    def getDropCol(self) -> int:
        user_input = input(f"Enter the column number to place disc for player {self.DiscColor.name} or enter 'Q' to exit: ")
        while not user_input.isdigit():
            ## If user enters escape, end the game
            if user_input.lower() == 'q':
                raise InterruptedError()
            user_input = input(f"Invalid input, please enter a number or 'Q' to exit: ")    
        return int(user_input)
    
class AIPlayer(Player):
    def getDropCol(self) -> int:
        pass #calculate col internally

class Board:
    def __init__(self, rows, cols, connectCount):
        self._rows = rows
        self._cols = cols
        self._connectCount = connectCount
        self._grid = [[None for _ in range(self._cols)] for _ in range(self._rows)] 

    def _hasWonVertically(self, col, row, color):
        counter = 1
        r = row - 1
        while r >= 0 and counter < self._connectCount:
            if self._grid[r][col] != color:
                break
            r -= 1
            counter += 1
            
        r = row + 1
        while r < self._rows and counter < 4:
            if self._grid[r][col] != color:
                break
            r += 1
            counter += 1
            
        return counter == self._connectCount

    def _hasWonHorizontally(self, col, row, color):
        counter = 1
        c = col - 1
        while c >= 0 and counter < self._connectCount:
            if self._grid[row][c] != color:
                break
            c -= 1
            counter += 1
            
        c = col + 1
        while c < self._cols and counter < 4:
            if self._grid[row][c] != color:
                break
            c += 1
            counter += 1
            
        return counter == self._connectCount

    def _hasWonDiagonallyAsc(self, col, row, color):
        counter = 1
        c = col - 1
        r = row - 1
        while r >= 0 and c >= 0 and counter < self._connectCount:
            if self._grid[r][c] != color:
                break
            r -= 1
            c -= 1
            counter += 1

        c = col + 1    
        r = row + 1
        while r < self._rows and c < self._cols and counter < 4:
            if self._grid[r][c] != color:
                break
            r += 1
            c += 1
            counter += 1
            
        return counter == self._connectCount

    def _hasWonDiagonallyDsc(self, col, row, color):
        counter = 1
        c = col - 1
        r = row + 1
        while r < self._rows and c >= 0 and counter < self._connectCount:
            if self._grid[r][c] != color:
                break
            r += 1
            c -= 1
            counter += 1

        c = col + 1    
        r = row - 1
        while r >= 0 and c < self._cols and counter < self._connectCount:
            if self._grid[r][c] != color:
                break
            r -= 1
            c += 1
            counter += 1
            
        return counter == self._connectCount

    def checkWin(self, col, row, color):    
        pass
        if col < 0 or row < 0 or col >= self._cols or row >= self._rows or color == None:
            raise ValueError('Invalid input') 
        return self._hasWonVertically(col, row, color) or self._hasWonHorizontally(col, row, color) or self._hasWonDiagonallyAsc(col, row, color) or self._hasWonDiagonallyDsc(col, row, color)
    
    def canDrop(self, col) -> bool:
        if self._grid[0][col] != None: #column is full
            return False
        return True
    
    def display(self):
        print('==== BOARD ====')
        print('')
        for r in range(self._rows):
            line = ""
            for c in range(self._cols):
                if self._grid[r][c] == DiscColor.YELLOW:
                    line += "Y"
                elif self._grid[r][c] == DiscColor.RED:
                    line += "R"
                else:
                    line += "-"
            print(line)
        print('')

    def dropDisc(self, col, discColor) -> int:
        if col < 0 or col >= self._cols:
            raise ValueError("Invalid value for columns")
        if not self.canDrop(col):
            return -1
        
        for r in range(self._rows-1,0,-1):
            if self._grid[r][col] == None:
                self._grid[r][col] = discColor
                return r
            
        return -1
    
    def isFull(self) -> bool:
        for c in range(self._cols):
            if self.canDrop(c):
                return False
            
        return True


FEEDBACK

What went well

Your bidirectional counting from the placed disc across all four orientations is exactly the right algorithm, and capping counter at 4 with early break keeps it efficient. 
You fixed the diagonal boundary guards so both _has4DiagonallyAsc() and _has4DiagonallyDsc() now correctly check c >= 0 and r against bounds, preventing the negative-index bug from before. 
Validating inputs in checkWin() before running any directional check is a clean way to reject bad positions up front. 

Minor suggestions

Your method names _has4Vertically and _has4Horizontally describe the opposite of what they do (the 'vertical' one varies col) — aligning them would make the code read more clearly. 

In [ ]:
# MY CORRECTION - should use direction vector in checkWin(), since the behavior is the same for all directions, the difference is in the value for increment and decrement.
class Board_V2(Board):
    def _inBound(self, row, col) -> bool:
        if col < 0 or row < 0 or col >= self._cols or row >= self._rows:
            return False
        return True
    
    def _countInDirection(self, row, col, dr, dc, color):
        count = 0
        r, c= row + dr, col + dc
        while self._inBound(r,c) and self._grid[r][c] == color:
            count += 1
            r += dr
            c += dc
        return count
        
    def checkWin(self, col, row, color):    
        if not self._inBound(col, row) or color == None:
            raise ValueError('Invalid input') 
        
        directions = [(0,1), (1,0), (1,1), (-1,-1)]
        for dr,dc in directions:
            matchingDisc = 1
            matchingDisc += self._countInDirection(row, col, dr, dc, color)
            # opposite direction
            matchingDisc += self._countInDirection(row, col, -dr, -dc, color)

            if matchingDisc >= self._connectCount:
                return True

        return False

Now implement makeMove(col) on your Game class. This is the main entry point — it needs to validate the move, delegate to Board, update currentPlayer, check for a win or draw, and update state. Walk me through the full flow including all the cases where you'd return false.

In [6]:
# 1. makeMove() will first check the state of the game. If the state is not "in-progress", function will return false.
# 2. Get the disc color of the current player.
# 3. Call Board.dropDisc(). Here I will make a change to the original design, instead of returning bool from dropDisc(), I will return the row number where the latest disc is placed. If the column is full or column value is not valid, dropDisc() will return -1. This change is needed, because I need both the column and row value for the latest disc in order to do win detection later.
# 4. If Board.dropDisc() returns negative value, makeMove() should return false immediately. Otherwise, call Board.checkWin(). If checkWin() returns true, set game state to 'won' and set winner to currentPlayer. If checkWin() returns false, call Board.isBoardFull(). If isBoardFull() returns true, set game state to 'draw'.
# 5. Finally, update currentPlayer to the other player and return true.

class Game:
    def __init__(self, player1: Player, player2: Player):
        if player1.DiscColor == player2.DiscColor: 
            raise ValueError("Players cannot have the same DiscColor")

        self._board = Board_V2(rows=6, cols=7, connectCount=4)
        self._player1 = player1
        self._player2 = player2
        self._currentPlayer = self._player1
        self._winner : Player = None
        self._gameState = GameState.IN_PROGRESS

    def getState(self) -> GameState:
        return self._gameState
    
    def makeMove(self) -> bool:
        if self._gameState != GameState.IN_PROGRESS:
            return False
        
        discColor = self._currentPlayer.DiscColor
        try:
            col = self._currentPlayer.getDropCol()
        except(InterruptedError): 
            #user quits the game
            self._gameState = GameState.STOPPED
            print("GAME ENDED BY USER!")
            return False
        
        row = self._board.dropDisc(col, discColor)

        if row < 0 : return False
        self._board.display()

        if self._board.checkWin(col, row, discColor):
            self._gameState = GameState.WON
            self._winner = self._currentPlayer
            print(f"GAME OVER! Winner is {discColor.name}")
        elif self._board.isFull():
            self._gameState = GameState.DRAW
            print("GAME OVER! DRAW!")
        
        self._currentPlayer = self._player2 if self._currentPlayer == self._player1 else self._player1
        return True 
    
    

In [9]:

game = Game(HumanPlayer(DiscColor.RED), HumanPlayer(DiscColor.YELLOW))
while game.getState() == GameState.IN_PROGRESS:
    game.makeMove()

==== BOARD ====

-------
-------
-------
-------
-------
R------

==== BOARD ====

-------
-------
-------
-------
-------
RY-----

==== BOARD ====

-------
-------
-------
-------
-R-----
RY-----

==== BOARD ====

-------
-------
-------
-------
-R-----
RYY----

==== BOARD ====

-------
-------
-------
-------
-RR----
RYY----

==== BOARD ====

-------
-------
-------
-------
-RR----
RYYY---

==== BOARD ====

-------
-------
-------
--R----
-RR----
RYYY---

==== BOARD ====

-------
-------
-------
--R----
-RRY---
RYYY---

==== BOARD ====

-------
-------
-------
--RR---
-RRY---
RYYY---

==== BOARD ====

-------
-------
-------
--RR---
-RRY---
RYYYY--

GAME OVER! Winner is YELLOW


FEEDBACK

What went well

Your flow is spot-on: you check gameState first, delegate to dropDisc(), handle win/draw, then switch turns — exactly the right ordering. 
You nailed the turn-switch fix from last time — self._player2 if self._currentPlayer == self._player1 else self._player1 now correctly alternates both ways. 
You handle all three failure cases cleanly: game already over, and invalid/full column via the row < 0 check. 

Your Board class has _rows = 6 and _cols = 7 hardcoded as instance variables, and your Game class hardcodes DiscColor.RED and DiscColor.YELLOW for player1 and player2. If a product requirement came in to support variable board sizes (e.g., 5x4 or 8x7) and a 'Connect Five' win condition instead of four, what changes would be needed across your Board, Game, and the win-check methods like _has4Vertically()? How would you restructure the design so these are configurable without modifying the core logic?

MY ANSWER

"Instead of hardcoding the rows and cols, I will pass in these 2 values as parameters in the Board constructor. This way we can support variable board sizes. Same for the win condition, the number of disc required will be a parameter and I will assign it to a private property _connectCount inside the Board class. For all the win condition checks, I will check against this private property. The helper function names should be more general, so instead of _has4Vertically(), I will rename it to _hasWonVertically().

Similarly, the players in the Game can be assigned dynamically, by having them passed in as parameters in the Game constructor. "

FEEDBACK

What went well

You correctly identified exactly which classes need to change and why — making rows, cols, and connectCount constructor parameters on Board is precisely the right approach. The win-check methods already use _rows and _cols internally, so they'll work correctly for any board size with no further logic changes. 
You also spotted the coupling in Game.__init__() where players were hardcoded with specific colors, and proposed the right fix: injecting players as constructor parameters. This is a classic dependency injection pattern and it's exactly what makes code flexible and testable. 

Minor suggestions

One small thing to consider: you might want to add a validation inside Board.__init__() to check that connectCount is not larger than both rows and cols — otherwise a 'Connect 5' on a 4x4 board would be impossible to win and could cause a subtle bug. 

Imagine you want to add an AI opponent to Game. Currently makeMove() accepts a column from an external caller, and Player only holds a DiscColor. How would you extend your Player class and Game.makeMove() to support both a human player (who provides a column externally) and an AI player (who computes its own column internally), without changing the Game loop logic significantly? What if you wanted multiple AI difficulty levels?

MY ANSWER

I will turn the Player Class into abstract class and add abstract method getDropCol() which returns an integer value for the column. This allows me to have different implementations of the abstract class, that is HumanPlayer class and AIPlayer class. The HumanPlayer class will get the drop column from user input, while the AIPlayer class can calculate the column internally. The AIPlayer class can also have its own property for difficulty levels.

Instead of having col to pass as a parameter to the makeMove(), the value will be obtained from _currentPlayer.getDropCol(). 

FEEDBACK

What went well

You correctly identified that Player needs to become abstract with a getDropCol() method, and that HumanPlayer and AIPlayer become concrete subclasses — this is exactly the Strategy/polymorphism pattern and it's the right tool here. You explained the why clearly: different players compute their column differently, so that responsibility belongs in Player. 
You also correctly noted that Game.makeMove() no longer needs a col parameter — it calls self._currentPlayer.getDropCol() instead. This means the Game loop logic barely changes at all, which is exactly the goal. You showed good instinct for keeping changes minimal and localized. 

Minor suggestions

One small thing to think about: HumanPlayer.getDropCol() needs to block and wait for input, while AIPlayer.getDropCol() computes immediately — these are very different behaviors. In a real system you might want to think about whether getDropCol() should receive the current Board state as a parameter, so the AIPlayer can actually analyze the board to pick a smart column.

Suppose your Connect Four platform becomes multi-player and you need to run thousands of Game instances simultaneously on a server, with spectators able to observe a game's current state in real-time. Looking at your Game class — specifically makeMove(), _currentPlayer, _gameState, and _winner — what race conditions could occur, and how would you make the game state thread-safe? How would you design a spectator observation mechanism without polling?

MY ANSWER

If the platform is to allow spectators to observe the game's state in real-time, race conditions could happen when the spectators ask for the game's state while the game is updating its properties, such as game state and winner.

One fix to prevent this race condition is to put a lock on the updates (game state and winner). Another option is to use observer pattern. Whenever a change is happening in the game (for example a change in the game state or the winner), a notification will be sent to the observers or the spectators. This way, there will be no polling and avoid the race condition. To implement this mechanism, the Game class will need to keep a list of spectators. It will also need new functions to add spectators to the list and send notifications to them. A change is also needed in the makeMove(). After updating the game state or winner, the send notification function will be called about the event. 

FEEDBACK

What went well

You directly addressed the race condition this time — you correctly identified that spectators reading _gameState and _winner while makeMove() is writing to them can cause an inconsistent snapshot, and you proposed using a lock on those updates. That's exactly the core issue the question was probing for, and it's a big improvement from your previous attempt. 
You connected the Observer pattern to your actual Game class design — you specifically mentioned that Game would hold a list of spectators, have a method to add spectators, and that makeMove() would call the notification function after updating state. This shows you understand which class owns the responsibility and where in the flow the notification happens. 

Minor suggestions

One small thing to consider: the lock and the notification should work together — ideally you release the lock before notifying spectators, so observers aren't blocked waiting for the lock while makeMove() is still holding it. A common pattern is: acquire lock → update state → release lock → notify observers. This avoids potential deadlocks as your system scales.